# Closed Lost Analysis — New Business (Classic Pipeline)

QoQ trend analysis across 8 hypotheses + win-rate cohort analysis.

**Scope:** Classic pipeline | New Business | `QUALIFIED_DATE IS NOT NULL` | Excludes test companies & 'Port'

**Source:** `PORT_ANALYTICS_PROD.DWH`

In [ ]:
import snowflake.connector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

conn = snowflake.connector.connect(connection_name="default")
cur = conn.cursor()
cur.execute("USE SCHEMA PORT_ANALYTICS_PROD.DWH")

def run_query(sql):
    cur.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def ql(dt):
    return pd.Timestamp(dt).to_period('Q').strftime('%YQ%q')

# Base WHERE clause used in all queries
BASE_FILTER = """
    f.IS_CLOSED = TRUE
    AND f.IS_WON = FALSE
    AND f.ARCHIVED = FALSE
    AND f._DELETED_TIMESTAMP IS NULL
    AND f.PIPELINE = 'Classic'
    AND f.DEAL_TYPE = 'newbusiness'
    AND f.QUALIFIED_DATE IS NOT NULL
    AND f.DEAL_CLOSED_DATE >= '2023-01-01'
    AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
    AND COALESCE(c.COMPANY_NAME, '') != 'Port'
"""

print("Connected — run all cells to generate charts.")

## 1. Region / Team Concentration

In [ ]:
df = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(f.DEAL_TEAM_NAME, 'Unassigned') AS team,
    COUNT(*) AS deals_lost,
    SUM(COALESCE(f.DEAL_TOTAL_ARR, 0)) AS arr_lost
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1, 2
ORDER BY 1, 2
""")
df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)

top_teams = ['US MM', 'US Ent', 'EMEA MM', 'EMEA Ent', 'APJ', 'Unassigned']
df_t = df[df['TEAM'].isin(top_teams)]
pivot = df_t.pivot_table(index='qlabel', columns='TEAM', values='DEALS_LOST', fill_value=0).reindex(sorted(df_t['qlabel'].unique()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot[top_teams].plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10')
axes[0].set_title('Closed Lost by Team (Absolute)', fontweight='bold')
axes[0].set_ylabel('# Deals Lost'); axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=7, loc='upper left')

pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct[top_teams].plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10')
axes[1].set_title('Closed Lost by Team (% Mix)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=7, loc='upper left')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); plt.show()

## 2. Dead Deals / Long Dwell Time (by Segment)

In [ ]:
df = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(c.SEGMENT, 'Other') AS segment,
    COUNT(*) AS total_deals,
    AVG(DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE)) AS avg_dwell_days,
    MEDIAN(DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE)) AS median_dwell_days,
    COUNT(CASE WHEN DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE) > 180 THEN 1 END) AS dead_deals_180d,
    ROUND(100.0 * COUNT(CASE WHEN DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE) > 180 THEN 1 END) / COUNT(*), 1) AS pct_dead_180d
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1, 2 ORDER BY 1, 2
""")
df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (seg, color) in enumerate([('Enterprise', '#e74c3c'), ('Mid-Market', '#3498db')]):
    ds = df[df['SEGMENT'] == seg].sort_values('CLOSE_QUARTER')
    axes[0, i].plot(ds['qlabel'], ds['AVG_DWELL_DAYS'], marker='o', label='Avg', color=color)
    axes[0, i].plot(ds['qlabel'], ds['MEDIAN_DWELL_DAYS'], marker='s', label='Median', color=color, alpha=0.5, linestyle='--')
    axes[0, i].set_title(f'{seg} — Dwell Time (days)', fontweight='bold')
    axes[0, i].set_ylabel('Days'); axes[0, i].tick_params(axis='x', rotation=45)
    axes[0, i].legend(fontsize=8); axes[0, i].grid(axis='y', alpha=0.3)

    axes[1, i].bar(ds['qlabel'], ds['DEAD_DEALS_180D'], color=color, alpha=0.7, label='Dead (>180d)')
    ax2 = axes[1, i].twinx()
    ax2.plot(ds['qlabel'], ds['PCT_DEAD_180D'], marker='D', color='#2c3e50', linewidth=2, label='%')
    axes[1, i].set_title(f'{seg} — Dead Deals (>180d)', fontweight='bold')
    axes[1, i].set_ylabel('#'); ax2.set_ylabel('%')
    axes[1, i].tick_params(axis='x', rotation=45)
    l1, la1 = axes[1, i].get_legend_handles_labels(); l2, la2 = ax2.get_legend_handles_labels()
    axes[1, i].legend(l1+l2, la1+la2, fontsize=8)
plt.tight_layout(); plt.show()

## 3. ARR Shrinkage (ARR at Loss / ARR at Qualification)

In [ ]:
df = run_query(f"""
WITH amount_at_qualify AS (
    -- First amount value recorded on or after the qualified date
    SELECT h.DEALID, TRY_TO_DECIMAL(h.VALUE, 15, 2) AS arr_at_qualify,
           ROW_NUMBER() OVER (PARTITION BY h.DEALID ORDER BY h.TIMESTAMP ASC) AS rn
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY h
    INNER JOIN FACT_DEALS f ON h.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER)
    WHERE h.PROPERTY = 'amount'
      AND h.TIMESTAMP >= f.QUALIFIED_DATE
      AND TRY_TO_DECIMAL(h.VALUE, 15, 2) IS NOT NULL
      AND TRY_TO_DECIMAL(h.VALUE, 15, 2) > 0
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS deals_with_both,
    AVG(f.DEAL_TOTAL_ARR) AS avg_arr_at_loss,
    AVG(aq.arr_at_qualify) AS avg_arr_at_qualify,
    AVG(CASE WHEN aq.arr_at_qualify > 0 THEN f.DEAL_TOTAL_ARR / aq.arr_at_qualify END) AS avg_ratio_loss_to_qualify
FROM FACT_DEALS f
INNER JOIN amount_at_qualify aq ON aq.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER) AND aq.rn = 1
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1
ORDER BY 1
""")

# Fallback: if property history is sparse, also show overall avg deal size trend
df_overall = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_deals,
    AVG(COALESCE(f.DEAL_TOTAL_ARR, 0)) AS avg_arr_at_loss,
    MEDIAN(COALESCE(f.DEAL_TOTAL_ARR, 0)) AS median_arr_at_loss
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1 ORDER BY 1
""")
df_overall['CLOSE_QUARTER'] = pd.to_datetime(df_overall['CLOSE_QUARTER'])
df_overall['qlabel'] = df_overall['CLOSE_QUARTER'].apply(ql)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: avg ARR at loss trend (all qualified lost deals)
axes[0].plot(df_overall['qlabel'], df_overall['AVG_ARR_AT_LOSS']/1000, marker='o', color='#e74c3c', label='Avg ARR at Loss ($K)')
axes[0].plot(df_overall['qlabel'], df_overall['MEDIAN_ARR_AT_LOSS']/1000, marker='s', color='#3498db', label='Median ARR at Loss ($K)')
axes[0].set_title('Avg Deal Size at Close-Lost (Qualified Deals)', fontweight='bold')
axes[0].set_ylabel('$K'); axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

# Right: ratio (ARR at loss / ARR at qualify) if data exists
if len(df) > 1:
    df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
    df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)
    axes[1].bar(df['qlabel'], df['AVG_RATIO_LOSS_TO_QUALIFY'], color='#8e44ad', alpha=0.7)
    axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
    axes[1].set_title('ARR at Loss / ARR at Qualify (ratio)', fontweight='bold')
    axes[1].set_ylabel('Ratio (1.0 = no change)')
    axes[1].tick_params(axis='x', rotation=45)
else:
    axes[1].text(0.5, 0.5, 'Insufficient property history\nfor ratio calculation',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    axes[1].set_title('ARR Ratio (limited data)', fontweight='bold')

plt.tight_layout(); plt.show()
print(f"Ratio data covers {len(df)} quarter(s) with {df['DEALS_WITH_BOTH'].sum() if len(df) > 0 else 0} deals.")

## 4. Owner Changes (Post-SDR)

In [ ]:
df = run_query(f"""
WITH stage_exit AS (
    SELECT DEALID, MIN(TIMESTAMP) AS post_sdr_ts
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY
    WHERE PROPERTY = 'dealstage'
      AND VALUE NOT IN ('65800978', '134696621', 'closedlost', 'closedwon')
    GROUP BY 1
),
owner_changes AS (
    SELECT h.DEALID, COUNT(*) AS owner_change_count
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY h
    INNER JOIN stage_exit se ON se.DEALID = h.DEALID
    WHERE h.PROPERTY = 'hubspot_owner_id' AND h.TIMESTAMP >= se.post_sdr_ts
    GROUP BY 1
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_deals,
    COUNT(oc.DEALID) AS deals_with_owner_change,
    ROUND(100.0 * COUNT(oc.DEALID) / COUNT(*), 1) AS pct_owner_changed,
    SUM(CASE WHEN COALESCE(oc.owner_change_count, 0) >= 2 THEN 1 ELSE 0 END) AS deals_with_2plus_changes
FROM FACT_DEALS f
LEFT JOIN owner_changes oc ON oc.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER)
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1 ORDER BY 1
""")
df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(df['qlabel'], df['PCT_OWNER_CHANGED'], color='#e67e22', alpha=0.7)
axes[0].set_title('% of Lost Deals with Owner Change (post-SDR)', fontweight='bold')
axes[0].set_ylabel('%'); axes[0].tick_params(axis='x', rotation=45); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df['qlabel'], df['DEALS_WITH_2PLUS_CHANGES'], color='#c0392b', alpha=0.7)
axes[1].set_title('Deals with 2+ Owner Changes (post-SDR)', fontweight='bold')
axes[1].set_ylabel('#'); axes[1].tick_params(axis='x', rotation=45); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Competitive Losses

In [ ]:
df = run_query(f"""
WITH evaluated_comp AS (
    SELECT DISTINCT DEALID
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY
    WHERE PROPERTY IN ('evaluated_competitors', 'competition')
      AND VALUE IS NOT NULL AND VALUE != '' AND LOWER(VALUE) NOT IN ('none', 'unknown')
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_lost,
    SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' THEN 1 ELSE 0 END) AS reason_competitive,
    COUNT(ec.DEALID) AS had_evaluated_competitor,
    SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' OR ec.DEALID IS NOT NULL THEN 1 ELSE 0 END) AS competitive_combined,
    ROUND(100.0 * SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' OR ec.DEALID IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_competitive,
    SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' OR ec.DEALID IS NOT NULL THEN COALESCE(f.DEAL_TOTAL_ARR, 0) ELSE 0 END) AS competitive_arr_lost
FROM FACT_DEALS f
LEFT JOIN evaluated_comp ec ON ec.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER)
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1 ORDER BY 1
""")
df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1 = axes[0]
ax1.bar(df['qlabel'], df['COMPETITIVE_COMBINED'], color='#e74c3c', alpha=0.7, label='Combined')
ax1.bar(df['qlabel'], df['REASON_COMPETITIVE'], color='#f39c12', alpha=0.9, label='Reason only')
ax1b = ax1.twinx()
ax1b.plot(df['qlabel'], df['PCT_COMPETITIVE'], marker='D', color='#2c3e50', linewidth=2, label='%')
ax1.set_title('Competitive Losses', fontweight='bold')
ax1.set_ylabel('#'); ax1b.set_ylabel('%'); ax1.tick_params(axis='x', rotation=45)
l1, la1 = ax1.get_legend_handles_labels(); l2, la2 = ax1b.get_legend_handles_labels()
ax1.legend(l1+l2, la1+la2, fontsize=8)

axes[1].bar(df['qlabel'], df['COMPETITIVE_ARR_LOST'].astype(float)/1e6, color='#8e44ad', alpha=0.7)
axes[1].set_title('Competitive ARR Lost ($M)', fontweight='bold')
axes[1].set_ylabel('$M'); axes[1].tick_params(axis='x', rotation=45); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Mega Source Mix Shift

In [ ]:
df = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(f.MEGA_SOURCE, 'Unknown') AS mega_source,
    COUNT(*) AS deals_lost,
    SUM(COALESCE(f.DEAL_TOTAL_ARR, 0)) AS arr_lost
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1, 2 ORDER BY 1, 2
""")
df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)

sources = ['Inbound', 'Outbount SDR', 'Outbound Sales', 'Channel', 'Referrals', 'Unknown']
pivot = df.pivot_table(index='qlabel', columns='MEGA_SOURCE', values='DEALS_LOST', fill_value=0).reindex(sorted(df['qlabel'].unique()))
cols = [c for c in sources if c in pivot.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot[cols].plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2')
axes[0].set_title('Closed Lost by Mega Source (Absolute)', fontweight='bold')
axes[0].set_ylabel('#'); axes[0].tick_params(axis='x', rotation=45); axes[0].legend(fontsize=7)

pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct[cols].plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('Closed Lost by Mega Source (% Mix)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].tick_params(axis='x', rotation=45); axes[1].legend(fontsize=7)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); plt.show()

## 7. Qualification Bar Loosening

In [ ]:
df = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_deals,
    AVG(DATEDIFF('day', f.DEAL_CREATED_DATE, f.QUALIFIED_DATE)) AS avg_days_to_qualify,
    COUNT(CASE WHEN DATEDIFF('day', f.DEAL_CREATED_DATE, f.QUALIFIED_DATE) <= 3 THEN 1 END) AS fast_qualified_3d,
    ROUND(100.0 * COUNT(CASE WHEN DATEDIFF('day', f.DEAL_CREATED_DATE, f.QUALIFIED_DATE) <= 3 THEN 1 END) / COUNT(*), 1) AS pct_fast_qualified,
    AVG(DATEDIFF('day', f.QUALIFIED_DATE, f.DEAL_CLOSED_DATE)) AS avg_days_qualify_to_close
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1 ORDER BY 1
""")
df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df['qlabel'], df['AVG_DAYS_TO_QUALIFY'], marker='o', color='#e74c3c', linewidth=2)
axes[0].set_title('Avg Days Create → Qualify (Lost Deals)', fontweight='bold')
axes[0].set_ylabel('Days'); axes[0].tick_params(axis='x', rotation=45); axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(df['qlabel'], df['PCT_FAST_QUALIFIED'], marker='s', color='#8e44ad', linewidth=2)
axes[1].fill_between(range(len(df)), df['PCT_FAST_QUALIFIED'], alpha=0.15, color='#8e44ad')
axes[1].set_title('% Qualified in <=3 Days (rubber-stamped?)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].set_xticks(range(len(df)))
axes[1].set_xticklabels(df['qlabel'], rotation=45, ha='right'); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Stage Before Close Mix Shift

In [ ]:
stage_map = {'65800978':'SDR Discovery','65800980':'Demo / Presentation','982622489':'Business Validation',
             '65537604':'Formal Pilot','134696621':'SDR / Omitted Opp','22339760':'Opportunity',
             '65537605':'Business Case Confirmation','134696626':'Negotiation / Legal',
             'contractsent':'Contract Sent','Unknown':'Unknown'}

df = run_query(f"""
WITH last_stage AS (
    SELECT h.DEALID, h.VALUE AS stage_before_close,
           ROW_NUMBER() OVER (PARTITION BY h.DEALID ORDER BY h.TIMESTAMP DESC) AS rn
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY h
    WHERE h.PROPERTY = 'dealstage' AND h.VALUE NOT IN ('closedlost', 'closedwon')
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(ls.stage_before_close, 'Unknown') AS stage_before_close,
    COUNT(*) AS deals_lost
FROM FACT_DEALS f
LEFT JOIN last_stage ls ON ls.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER) AND ls.rn = 1
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {BASE_FILTER}
GROUP BY 1, 2 ORDER BY 1, 3 DESC
""")
df['CLOSE_QUARTER'] = pd.to_datetime(df['CLOSE_QUARTER'])
df['qlabel'] = df['CLOSE_QUARTER'].apply(ql)
df['stage_label'] = df['STAGE_BEFORE_CLOSE'].map(stage_map).fillna(df['STAGE_BEFORE_CLOSE'])

pivot = df.pivot_table(index='qlabel', columns='stage_label', values='DEALS_LOST', fill_value=0).reindex(sorted(df['qlabel'].unique()))
main = [c for c in ['SDR Discovery','Demo / Presentation','Business Validation','Formal Pilot',
        'Business Case Confirmation','Negotiation / Legal','Unknown'] if c in pivot.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot[main].plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis')
axes[0].set_title('Stage Before Close-Lost (Absolute)', fontweight='bold')
axes[0].set_ylabel('#'); axes[0].tick_params(axis='x', rotation=45); axes[0].legend(fontsize=7, loc='upper left')

pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct[main].plot(kind='bar', stacked=True, ax=axes[1], colormap='viridis')
axes[1].set_title('Stage Before Close-Lost (% Mix)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].tick_params(axis='x', rotation=45); axes[1].legend(fontsize=7, loc='upper left')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); plt.show()

## 9. Conversion Rate (Create → Won) by Cohort Quarter

Win rate = deals closed-won / all deals created in that quarter (qualified, Classic, new business).

Broken down by **Team**, **Mega Source**, and **ARR Bucket**.

In [ ]:
# CVR base filter: all deals (not just lost) that are qualified, classic, new biz
CVR_FILTER = """
    f.ARCHIVED = FALSE
    AND f._DELETED_TIMESTAMP IS NULL
    AND f.PIPELINE = 'Classic'
    AND f.DEAL_TYPE = 'newbusiness'
    AND f.QUALIFIED_DATE IS NOT NULL
    AND f.DEAL_CREATED_DATE >= '2023-01-01'
    AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
    AND COALESCE(c.COMPANY_NAME, '') != 'Port'
"""

# --- Overall CVR ---
cvr_overall = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CREATED_DATE)::DATE AS create_quarter,
    COUNT(*) AS deals_created,
    SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) AS deals_won,
    ROUND(100.0 * SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {CVR_FILTER}
GROUP BY 1 ORDER BY 1
""")
cvr_overall['CREATE_QUARTER'] = pd.to_datetime(cvr_overall['CREATE_QUARTER'])
cvr_overall['qlabel'] = cvr_overall['CREATE_QUARTER'].apply(ql)

# --- CVR by Team ---
cvr_team = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CREATED_DATE)::DATE AS create_quarter,
    COALESCE(f.DEAL_TEAM_NAME, 'Unassigned') AS team,
    COUNT(*) AS deals_created,
    SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) AS deals_won,
    ROUND(100.0 * SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {CVR_FILTER}
GROUP BY 1, 2 ORDER BY 1, 2
""")
cvr_team['CREATE_QUARTER'] = pd.to_datetime(cvr_team['CREATE_QUARTER'])
cvr_team['qlabel'] = cvr_team['CREATE_QUARTER'].apply(ql)

# --- CVR by Mega Source ---
cvr_source = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CREATED_DATE)::DATE AS create_quarter,
    COALESCE(f.MEGA_SOURCE, 'Unknown') AS mega_source,
    COUNT(*) AS deals_created,
    SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) AS deals_won,
    ROUND(100.0 * SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {CVR_FILTER}
GROUP BY 1, 2 ORDER BY 1, 2
""")
cvr_source['CREATE_QUARTER'] = pd.to_datetime(cvr_source['CREATE_QUARTER'])
cvr_source['qlabel'] = cvr_source['CREATE_QUARTER'].apply(ql)

# --- CVR by ARR Bucket ---
cvr_arr = run_query(f"""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CREATED_DATE)::DATE AS create_quarter,
    CASE
        WHEN COALESCE(f.DEAL_TOTAL_ARR, 0) < 10000 THEN '<$10K'
        WHEN COALESCE(f.DEAL_TOTAL_ARR, 0) BETWEEN 10000 AND 50000 THEN '$10K-$50K'
        ELSE '>$50K'
    END AS arr_bucket,
    COUNT(*) AS deals_created,
    SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) AS deals_won,
    ROUND(100.0 * SUM(CASE WHEN f.IS_CLOSED = TRUE AND f.IS_WON = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE {CVR_FILTER}
GROUP BY 1, 2 ORDER BY 1, 2
""")
cvr_arr['CREATE_QUARTER'] = pd.to_datetime(cvr_arr['CREATE_QUARTER'])
cvr_arr['qlabel'] = cvr_arr['CREATE_QUARTER'].apply(ql)

print(f"Data loaded: {len(cvr_overall)} quarters, team={len(cvr_team)} rows, source={len(cvr_source)} rows, arr={len(cvr_arr)} rows")

In [ ]:
# --- Plot CVR charts ---
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Overall win rate
axes[0, 0].plot(cvr_overall['qlabel'], cvr_overall['WIN_RATE'], marker='o', color='#27ae60', linewidth=2)
axes[0, 0].fill_between(range(len(cvr_overall)), cvr_overall['WIN_RATE'], alpha=0.15, color='#27ae60')
axes[0, 0].set_title('Overall Win Rate by Create Quarter', fontweight='bold')
axes[0, 0].set_ylabel('Win Rate %'); axes[0, 0].set_xticks(range(len(cvr_overall)))
axes[0, 0].set_xticklabels(cvr_overall['qlabel'], rotation=45, ha='right')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. By Team
top_cvr_teams = ['US MM', 'US Ent', 'EMEA MM', 'EMEA Ent', 'APJ']
for team in top_cvr_teams:
    dt = cvr_team[cvr_team['TEAM'] == team].sort_values('CREATE_QUARTER')
    if len(dt) > 2:
        axes[0, 1].plot(dt['qlabel'], dt['WIN_RATE'], marker='o', label=team, linewidth=1.5)
axes[0, 1].set_title('Win Rate by Team', fontweight='bold')
axes[0, 1].set_ylabel('Win Rate %'); axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].legend(fontsize=7); axes[0, 1].grid(axis='y', alpha=0.3)

# 3. By Mega Source
for src in ['Inbound', 'Outbount SDR', 'Outbound Sales']:
    ds = cvr_source[cvr_source['MEGA_SOURCE'] == src].sort_values('CREATE_QUARTER')
    if len(ds) > 2:
        axes[1, 0].plot(ds['qlabel'], ds['WIN_RATE'], marker='o', label=src, linewidth=1.5)
axes[1, 0].set_title('Win Rate by Mega Source', fontweight='bold')
axes[1, 0].set_ylabel('Win Rate %'); axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].legend(fontsize=8); axes[1, 0].grid(axis='y', alpha=0.3)

# 4. By ARR Bucket
for bucket in ['<$10K', '$10K-$50K', '>$50K']:
    db = cvr_arr[cvr_arr['ARR_BUCKET'] == bucket].sort_values('CREATE_QUARTER')
    if len(db) > 2:
        axes[1, 1].plot(db['qlabel'], db['WIN_RATE'], marker='o', label=bucket, linewidth=1.5)
axes[1, 1].set_title('Win Rate by ARR Bucket', fontweight='bold')
axes[1, 1].set_ylabel('Win Rate %'); axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].legend(fontsize=8); axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

# Closed Lost Analysis — New Business (Classic Pipeline)

QoQ trend analysis across 8 hypotheses for increasing closed-lost deals.

**Filters:** Classic pipeline | New Business | Excludes test companies & 'Port' | Source: `PORT_ANALYTICS_PROD.DWH`

**Data pulled:** 2026-07-06

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

def quarter_label(dt):
    return pd.Timestamp(dt).to_period('Q').strftime('%YQ%q')

print("Setup complete — data embedded as snapshots (2026-07-06)")

## 1. Region / Team Concentration

In [ ]:
region_team_data = pd.DataFrame([
    ("2023-01-01","Unassigned",59,280201),("2023-04-01","Unassigned",57,474740),
    ("2023-07-01","Exec",16,415555),("2023-07-01","Unassigned",59,475400),
    ("2023-10-01","EMEA Ent",4,0),("2023-10-01","Exec",34,251120),("2023-10-01","US Ent",17,454000),("2023-10-01","US MM",24,481200),("2023-10-01","Unassigned",19,80000),
    ("2024-01-01","EMEA Ent",21,420000),("2024-01-01","EMEA MM",16,30000),("2024-01-01","Exec",15,137000),("2024-01-01","US Ent",37,1500200),("2024-01-01","US MM",40,1014500),("2024-01-01","Unassigned",26,0),
    ("2024-04-01","EMEA Ent",46,1070000),("2024-04-01","EMEA MM",49,285000),("2024-04-01","US Ent",30,1120000),("2024-04-01","US MM",33,626000),("2024-04-01","Unassigned",5,70000),
    ("2024-07-01","EMEA Ent",23,675000),("2024-07-01","EMEA MM",35,597800),("2024-07-01","US Ent",15,455000),("2024-07-01","US MM",41,1066600),
    ("2024-10-01","EMEA Ent",29,814500),("2024-10-01","EMEA MM",41,471400),("2024-10-01","US Ent",52,778675),("2024-10-01","US MM",43,565000),
    ("2025-01-01","APJ",19,255000),("2025-01-01","EMEA Ent",23,410000),("2025-01-01","EMEA MM",51,541000),("2025-01-01","US Ent",58,1518000),("2025-01-01","US MM",91,2040240),
    ("2025-04-01","APJ",12,782000),("2025-04-01","EMEA Ent",24,447000),("2025-04-01","EMEA MM",43,559860),("2025-04-01","US Ent",63,1717710),("2025-04-01","US MM",48,1110000),("2025-04-01","Unassigned",54,0),
    ("2025-07-01","APJ",15,109000),("2025-07-01","EMEA Ent",24,1160000),("2025-07-01","EMEA MM",34,597200),("2025-07-01","US Ent",42,935000),("2025-07-01","US MM",41,1164400),("2025-07-01","Unassigned",51,0),
    ("2025-10-01","APJ",10,343000),("2025-10-01","EMEA Ent",37,875000),("2025-10-01","EMEA MM",35,540040),("2025-10-01","US Ent",34,886000),("2025-10-01","US MM",47,1878875),("2025-10-01","Unassigned",21,0),
    ("2026-01-01","APJ",14,470400),("2026-01-01","EMEA Ent",36,1147000),("2026-01-01","EMEA MM",32,1063000),("2026-01-01","US Ent",83,4858000),("2026-01-01","US MM",49,1146650),("2026-01-01","Unassigned",25,0),
    ("2026-04-01","APJ",19,349950),("2026-04-01","EMEA Ent",87,2400000),("2026-04-01","EMEA MM",92,1587000),("2026-04-01","US Ent",78,4250000),("2026-04-01","US MM",63,2847600),("2026-04-01","Unassigned",101,0),
], columns=["CLOSE_QUARTER","TEAM","DEALS_LOST","ARR_LOST"])
region_team_data['CLOSE_QUARTER'] = pd.to_datetime(region_team_data['CLOSE_QUARTER'])
region_team_data['qlabel'] = region_team_data['CLOSE_QUARTER'].apply(quarter_label)

top_teams = ['US MM', 'US Ent', 'EMEA MM', 'EMEA Ent', 'APJ', 'Unassigned']
df_t = region_team_data[region_team_data['TEAM'].isin(top_teams)]
pivot = df_t.pivot_table(index='qlabel', columns='TEAM', values='DEALS_LOST', fill_value=0).reindex(sorted(df_t['qlabel'].unique()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot[top_teams].plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10')
axes[0].set_title('Closed Lost by Team (Absolute)', fontweight='bold'); axes[0].set_ylabel('# Deals Lost')
axes[0].tick_params(axis='x', rotation=45); axes[0].legend(fontsize=7, loc='upper left')

pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct[top_teams].plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10')
axes[1].set_title('Closed Lost by Team (% Mix)', fontweight='bold'); axes[1].set_ylabel('% of Deals Lost')
axes[1].tick_params(axis='x', rotation=45); axes[1].legend(fontsize=7, loc='upper left')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); plt.show()

## 2. Dead Deals / Long Dwell Time (by Segment)

In [ ]:
dwell_data = pd.DataFrame([
    ("2023-01-01","Enterprise",8,56.4,58.0,0,0.0),("2023-01-01","Mid-Market",9,53.6,58.0,0,0.0),
    ("2023-04-01","Enterprise",13,52.2,41.0,0,0.0),("2023-04-01","Mid-Market",15,95.7,69.0,2,13.3),
    ("2023-07-01","Enterprise",20,85.0,85.5,1,5.0),("2023-07-01","Mid-Market",19,80.5,77.0,0,0.0),
    ("2023-10-01","Enterprise",34,81.5,77.5,2,5.9),("2023-10-01","Mid-Market",54,78.2,49.5,8,14.8),
    ("2024-01-01","Enterprise",61,102.8,87.0,7,11.5),("2024-01-01","Mid-Market",91,73.5,66.0,5,5.5),
    ("2024-04-01","Enterprise",74,66.3,48.5,4,5.4),("2024-04-01","Mid-Market",83,55.1,38.0,4,4.8),
    ("2024-07-01","Enterprise",38,78.3,51.0,4,10.5),("2024-07-01","Mid-Market",73,60.6,51.0,4,5.5),
    ("2024-10-01","Enterprise",76,80.7,42.0,8,10.5),("2024-10-01","Mid-Market",86,67.2,36.5,11,12.8),
    ("2025-01-01","Enterprise",85,84.4,63.0,9,10.6),("2025-01-01","Mid-Market",166,72.1,57.0,8,4.8),
    ("2025-04-01","Enterprise",104,90.6,61.5,13,12.5),("2025-04-01","Mid-Market",157,72.1,63.0,12,7.6),
    ("2025-07-01","Enterprise",108,103.9,80.5,12,11.1),("2025-07-01","Mid-Market",124,108.6,93.5,23,18.5),
    ("2025-10-01","Enterprise",74,98.7,68.0,13,17.6),("2025-10-01","Mid-Market",110,95.9,85.0,18,16.4),
    ("2026-01-01","Enterprise",143,168.9,98.0,57,39.9),("2026-01-01","Mid-Market",120,108.8,63.5,25,20.8),
    ("2026-04-01","Enterprise",192,125.9,70.0,46,24.0),("2026-04-01","Mid-Market",197,58.3,28.0,13,6.6),
], columns=["CLOSE_QUARTER","SEGMENT","TOTAL_DEALS","AVG_DWELL_DAYS","MEDIAN_DWELL_DAYS","DEAD_DEALS_180D","PCT_DEAD_180D"])
dwell_data['CLOSE_QUARTER'] = pd.to_datetime(dwell_data['CLOSE_QUARTER'])
dwell_data['qlabel'] = dwell_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (seg, color) in enumerate([('Enterprise', '#e74c3c'), ('Mid-Market', '#3498db')]):
    df_seg = dwell_data[dwell_data['SEGMENT'] == seg].sort_values('CLOSE_QUARTER')
    axes[0, i].plot(df_seg['qlabel'], df_seg['AVG_DWELL_DAYS'], marker='o', label='Avg', color=color)
    axes[0, i].plot(df_seg['qlabel'], df_seg['MEDIAN_DWELL_DAYS'], marker='s', label='Median', color=color, alpha=0.5, linestyle='--')
    axes[0, i].set_title(f'{seg} — Dwell Time (days)', fontweight='bold')
    axes[0, i].set_ylabel('Days'); axes[0, i].tick_params(axis='x', rotation=45)
    axes[0, i].legend(fontsize=8); axes[0, i].grid(axis='y', alpha=0.3)

    axes[1, i].bar(df_seg['qlabel'], df_seg['DEAD_DEALS_180D'], color=color, alpha=0.7, label='Dead (>180d)')
    ax2 = axes[1, i].twinx()
    ax2.plot(df_seg['qlabel'], df_seg['PCT_DEAD_180D'], marker='D', color='#2c3e50', linewidth=2, label='% of total')
    axes[1, i].set_title(f'{seg} — Dead Deals (>180d)', fontweight='bold')
    axes[1, i].set_ylabel('# Dead Deals'); ax2.set_ylabel('%')
    axes[1, i].tick_params(axis='x', rotation=45)
    l1, la1 = axes[1, i].get_legend_handles_labels()
    l2, la2 = ax2.get_legend_handles_labels()
    axes[1, i].legend(l1+l2, la1+la2, fontsize=8)
plt.tight_layout(); plt.show()

## 3. ARR Shrinkage (Amount Changes in Deal History)

Note: Property history coverage for `amount` is limited — only 23 closed-lost deals have recorded amount changes (all in 2026 Q2+).

In [ ]:
arr_data = pd.DataFrame([
    ("2026-04-01", 22, 118425.0, 117686.4, -738.6, 102.1, 4, 18.2),
    ("2026-07-01", 1, 150000.0, 0.0, -150000.0, 0.0, 1, 100.0),
], columns=["CLOSE_QUARTER","TOTAL_DEALS_WITH_HISTORY","AVG_FIRST_AMOUNT","AVG_LAST_AMOUNT","AVG_ARR_CHANGE","AVG_PCT_OF_ORIGINAL","DEALS_SHRANK","PCT_DEALS_SHRANK"])
arr_data['CLOSE_QUARTER'] = pd.to_datetime(arr_data['CLOSE_QUARTER'])
arr_data['qlabel'] = arr_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(arr_data['qlabel'], arr_data['AVG_FIRST_AMOUNT']/1000, alpha=0.7, color='#27ae60', label='Avg First ($K)')
axes[0].bar(arr_data['qlabel'], arr_data['AVG_LAST_AMOUNT']/1000, alpha=0.7, color='#e74c3c', label='Avg Last ($K)', width=0.3)
axes[0].set_title('ARR: First vs Final Amount (deals with history)', fontweight='bold')
axes[0].set_ylabel('$K'); axes[0].legend()

axes[1].bar(arr_data['qlabel'], arr_data['DEALS_SHRANK'], color='#e74c3c', alpha=0.7)
axes[1].set_title(f'Deals that Shrank: {arr_data["DEALS_SHRANK"].sum()} of {arr_data["TOTAL_DEALS_WITH_HISTORY"].sum()}', fontweight='bold')
axes[1].set_ylabel('# Deals')
plt.tight_layout(); plt.show()
print("\\nLimited data: only 23 deals have amount-change history. Monitor as coverage grows.")

## 4. Owner Changes (Post-SDR)

Owner change history from `DEALS_PROPERTY_HISTORY` only has data for recent deals (2026 Q2+).

In [ ]:
owner_data = pd.DataFrame([
    ("2023-01-01",59,0,0.0,0),("2023-04-01",57,0,0.0,0),("2023-07-01",75,0,0.0,0),
    ("2023-10-01",98,0,0.0,0),("2024-01-01",155,0,0.0,0),("2024-04-01",168,0,0.0,0),
    ("2024-07-01",117,0,0.0,0),("2024-10-01",172,0,0.0,0),("2025-01-01",256,0,0.0,0),
    ("2025-04-01",271,0,0.0,0),("2025-07-01",237,0,0.0,0),("2025-10-01",190,0,0.0,0),
    ("2026-01-01",275,0,0.0,0),("2026-04-01",492,18,3.7,10),
], columns=["CLOSE_QUARTER","TOTAL_DEALS","DEALS_WITH_OWNER_CHANGE","PCT_OWNER_CHANGED","DEALS_WITH_2PLUS_CHANGES"])
owner_data['CLOSE_QUARTER'] = pd.to_datetime(owner_data['CLOSE_QUARTER'])
owner_data['qlabel'] = owner_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(owner_data['qlabel'], owner_data['PCT_OWNER_CHANGED'], color='#e67e22', alpha=0.7)
axes[0].set_title('% of Lost Deals with Owner Change (post-SDR)', fontweight='bold')
axes[0].set_ylabel('%'); axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(owner_data['qlabel'], owner_data['DEALS_WITH_2PLUS_CHANGES'], color='#c0392b', alpha=0.7)
axes[1].set_title('Deals with 2+ Owner Changes (post-SDR)', fontweight='bold')
axes[1].set_ylabel('# Deals'); axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print("\\nNote: Property history only covers recent deals. 2026 Q2 shows 18/492 (3.7%) with post-SDR owner changes, 10 with 2+.")

## 5. Competitive Losses

In [ ]:
comp_data = pd.DataFrame([
    ("2023-01-01",59,2,0,2,3.4,0),("2023-04-01",57,0,0,0,0.0,0),
    ("2023-07-01",75,5,0,5,6.7,160000),("2023-10-01",98,8,0,8,8.2,145000),
    ("2024-01-01",155,9,0,9,5.8,348000),("2024-04-01",168,14,0,14,8.3,749000),
    ("2024-07-01",117,9,1,10,8.5,463600),("2024-10-01",172,9,1,10,5.8,253000),
    ("2025-01-01",256,23,1,24,9.4,1271240),("2025-04-01",271,11,0,11,4.1,655000),
    ("2025-07-01",237,12,1,13,5.5,563000),("2025-10-01",190,9,0,9,4.7,858000),
    ("2026-01-01",275,9,3,12,4.4,1083000),("2026-04-01",492,23,16,32,6.5,4128550),
], columns=["CLOSE_QUARTER","TOTAL_LOST","REASON_COMPETITIVE","HAD_EVALUATED_COMPETITOR","COMPETITIVE_LOSSES_COMBINED","PCT_COMPETITIVE","COMPETITIVE_ARR_LOST"])
comp_data['CLOSE_QUARTER'] = pd.to_datetime(comp_data['CLOSE_QUARTER'])
comp_data['qlabel'] = comp_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1 = axes[0]
ax1.bar(comp_data['qlabel'], comp_data['COMPETITIVE_LOSSES_COMBINED'], color='#e74c3c', alpha=0.7, label='Combined')
ax1.bar(comp_data['qlabel'], comp_data['REASON_COMPETITIVE'], color='#f39c12', alpha=0.9, label='Reason only')
ax1b = ax1.twinx()
ax1b.plot(comp_data['qlabel'], comp_data['PCT_COMPETITIVE'], marker='D', color='#2c3e50', linewidth=2, label='% of Total')
ax1.set_title('Competitive Losses (Reason + Evaluated Competitor)', fontweight='bold')
ax1.set_ylabel('# Deals'); ax1b.set_ylabel('%')
ax1.tick_params(axis='x', rotation=45)
l1,la1=ax1.get_legend_handles_labels(); l2,la2=ax1b.get_legend_handles_labels()
ax1.legend(l1+l2, la1+la2, fontsize=8)

axes[1].bar(comp_data['qlabel'], comp_data['COMPETITIVE_ARR_LOST']/1e6, color='#8e44ad', alpha=0.7)
axes[1].set_title('Competitive ARR Lost ($M)', fontweight='bold')
axes[1].set_ylabel('$M'); axes[1].tick_params(axis='x', rotation=45); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Mega Source Mix Shift

In [ ]:
src_data = pd.DataFrame([
    ("2023-01-01","Inbound",10,280201),("2023-01-01","Unknown",49,0),
    ("2023-04-01","Inbound",57,474740),
    ("2023-07-01","Inbound",72,890955),("2023-07-01","Referrals",2,0),
    ("2023-10-01","Inbound",93,1226320),("2023-10-01","Referrals",5,40000),
    ("2024-01-01","Inbound",140,2835700),("2024-01-01","Outbound Sales",2,130000),("2024-01-01","Outbount SDR",10,36000),("2024-01-01","Referrals",3,100000),
    ("2024-04-01","Inbound",143,2748000),("2024-04-01","Outbound Sales",7,340000),("2024-04-01","Outbount SDR",14,18000),("2024-04-01","Referrals",4,65000),
    ("2024-07-01","Inbound",92,2513600),("2024-07-01","Outbound Sales",3,84000),("2024-07-01","Outbount SDR",18,120000),("2024-07-01","Referrals",4,76800),
    ("2024-10-01","Inbound",140,2243575),("2024-10-01","Outbound Sales",5,176000),("2024-10-01","Outbount SDR",18,18000),("2024-10-01","Referrals",9,192000),
    ("2025-01-01","Inbound",185,3509600),("2025-01-01","Outbound Sales",17,207000),("2025-01-01","Outbount SDR",46,861240),("2025-01-01","Referrals",7,186400),
    ("2025-04-01","Inbound",207,3105570),("2025-04-01","Outbound Sales",12,788000),("2025-04-01","Outbount SDR",45,576000),("2025-04-01","Referrals",4,0),("2025-04-01","Channel",3,147000),
    ("2025-07-01","Inbound",183,3454600),("2025-07-01","Outbound Sales",6,48000),("2025-07-01","Outbount SDR",36,413000),("2025-07-01","Referrals",6,50000),("2025-07-01","Channel",6,0),
    ("2025-10-01","Inbound",103,2936915),("2025-10-01","Outbound Sales",13,960000),("2025-10-01","Outbount SDR",54,364000),("2025-10-01","Channel",17,212000),
    ("2026-01-01","Inbound",177,6835650),("2026-01-01","Outbound Sales",11,270000),("2026-01-01","Outbount SDR",63,989000),("2026-01-01","Channel",19,590400),
    ("2026-04-01","Inbound",366,7523100),("2026-04-01","Outbound Sales",22,1054000),("2026-04-01","Outbount SDR",77,2081000),("2026-04-01","Channel",13,775000),("2026-04-01","Unknown",12,0),
], columns=["CLOSE_QUARTER","MEGA_SOURCE","DEALS_LOST","ARR_LOST"])
src_data['CLOSE_QUARTER'] = pd.to_datetime(src_data['CLOSE_QUARTER'])
src_data['qlabel'] = src_data['CLOSE_QUARTER'].apply(quarter_label)

sources = ['Inbound', 'Outbount SDR', 'Outbound Sales', 'Channel', 'Referrals', 'Unknown']
pivot_src = src_data.pivot_table(index='qlabel', columns='MEGA_SOURCE', values='DEALS_LOST', fill_value=0).reindex(sorted(src_data['qlabel'].unique()))
cols = [c for c in sources if c in pivot_src.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot_src[cols].plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2')
axes[0].set_title('Closed Lost by Mega Source (Absolute)', fontweight='bold')
axes[0].set_ylabel('# Deals'); axes[0].tick_params(axis='x', rotation=45); axes[0].legend(fontsize=7)

pivot_pct = pivot_src.div(pivot_src.sum(axis=1), axis=0) * 100
pivot_pct[cols].plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('Closed Lost by Mega Source (% Mix)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].tick_params(axis='x', rotation=45); axes[1].legend(fontsize=7)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); plt.show()

## 7. Qualification Bar Loosening

In [ ]:
qual_data = pd.DataFrame([
    ("2023-01-01",59,4,6.8,0.0,4,100.0),("2023-04-01",57,15,26.3,16.7,10,66.7),
    ("2023-07-01",75,27,36.0,17.7,9,33.3),("2023-10-01",98,35,35.7,13.8,10,28.6),
    ("2024-01-01",155,64,41.3,13.9,15,23.4),("2024-04-01",168,50,29.8,9.8,12,24.0),
    ("2024-07-01",117,49,41.9,19.8,5,10.2),("2024-10-01",172,46,26.7,13.8,11,23.9),
    ("2025-01-01",256,78,30.5,16.9,16,20.5),("2025-04-01",271,68,25.1,24.6,13,19.1),
    ("2025-07-01",237,58,24.5,22.4,6,10.3),("2025-10-01",190,56,29.5,13.7,12,21.4),
    ("2026-01-01",275,85,30.9,25.1,9,10.6),("2026-04-01",492,78,15.9,19.6,13,16.7),
], columns=["CLOSE_QUARTER","TOTAL_DEALS","QUALIFIED_DEALS","PCT_QUALIFIED","AVG_DAYS_TO_QUALIFY","FAST_QUALIFIED_3D","PCT_FAST_QUALIFIED"])
qual_data['CLOSE_QUARTER'] = pd.to_datetime(qual_data['CLOSE_QUARTER'])
qual_data['qlabel'] = qual_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1 = axes[0]
ax1.bar(qual_data['qlabel'], qual_data['PCT_QUALIFIED'], color='#27ae60', alpha=0.7, label='% Qualified')
ax1b = ax1.twinx()
ax1b.plot(qual_data['qlabel'], qual_data['AVG_DAYS_TO_QUALIFY'], marker='o', color='#e74c3c', linewidth=2, label='Avg Days')
ax1.set_title('Qualification Rate & Speed (Lost Deals)', fontweight='bold')
ax1.set_ylabel('% Qualified'); ax1b.set_ylabel('Days')
ax1.tick_params(axis='x', rotation=45)
l1,la1=ax1.get_legend_handles_labels(); l2,la2=ax1b.get_legend_handles_labels()
ax1.legend(l1+l2, la1+la2, fontsize=8)

axes[1].plot(qual_data['qlabel'], qual_data['PCT_FAST_QUALIFIED'], marker='s', color='#8e44ad', linewidth=2)
axes[1].fill_between(range(len(qual_data)), qual_data['PCT_FAST_QUALIFIED'], alpha=0.15, color='#8e44ad')
axes[1].set_title('% Qualified in <=3 Days (of those qualified)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].set_xticks(range(len(qual_data)))
axes[1].set_xticklabels(qual_data['qlabel'], rotation=45, ha='right'); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Stage Before Close Mix Shift

Note: Stage history coverage from `DEALS_PROPERTY_HISTORY` is only available for 2026 Q1+ deals.

In [ ]:
stage_map = {'65800978':'SDR Discovery','65800980':'Demo / Presentation','982622489':'Business Validation',
             '65537604':'Formal Pilot','134696621':'SDR / Omitted Opp','22339760':'Opportunity',
             '65537605':'Business Case Confirmation','134696626':'Negotiation / Legal',
             'contractsent':'Contract Sent','Unknown':'Unknown'}

# Only show quarters with resolved stage data (2026 Q1+)
stage_data = pd.DataFrame([
    ("2026-01-01","Unknown",266),("2026-01-01","65800978",9),
    ("2026-04-01","65800978",295),("2026-04-01","Unknown",152),("2026-04-01","982622489",19),
    ("2026-04-01","65800980",11),("2026-04-01","contractsent",5),("2026-04-01","65537604",5),
    ("2026-04-01","22339760",3),("2026-04-01","65537605",2),
], columns=["CLOSE_QUARTER","STAGE_BEFORE_CLOSE","DEALS_LOST"])
stage_data['CLOSE_QUARTER'] = pd.to_datetime(stage_data['CLOSE_QUARTER'])
stage_data['qlabel'] = stage_data['CLOSE_QUARTER'].apply(quarter_label)
stage_data['stage_label'] = stage_data['STAGE_BEFORE_CLOSE'].map(stage_map).fillna(stage_data['STAGE_BEFORE_CLOSE'])

pivot_st = stage_data.pivot_table(index='qlabel', columns='stage_label', values='DEALS_LOST', fill_value=0)
pivot_st = pivot_st.reindex(sorted(pivot_st.index))
main_stages = [c for c in ['SDR Discovery','Demo / Presentation','Business Validation',
               'Formal Pilot','Business Case Confirmation','Opportunity','Contract Sent','Unknown'] if c in pivot_st.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pivot_st[main_stages].plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis')
axes[0].set_title('Stage Before Close-Lost (Absolute, 2026 Q1+)', fontweight='bold')
axes[0].set_ylabel('# Deals'); axes[0].tick_params(axis='x', rotation=0); axes[0].legend(fontsize=7, loc='upper left')

pivot_pct = pivot_st.div(pivot_st.sum(axis=1), axis=0) * 100
pivot_pct[main_stages].plot(kind='bar', stacked=True, ax=axes[1], colormap='viridis')
axes[1].set_title('Stage Before Close-Lost (% Mix)', fontweight='bold')
axes[1].set_ylabel('%'); axes[1].tick_params(axis='x', rotation=0); axes[1].legend(fontsize=7, loc='upper left')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); plt.show()

print("\\n2026 Q2: 295/492 (60%) of lost deals died in SDR Discovery stage.")
print("Only 45 deals (9%) made it past Demo before closing lost.")

# Closed Lost Analysis — New Business (Classic Pipeline)

QoQ trend analysis across 8 hypotheses for why closed-lost deals are increasing.

**Filters:** Classic pipeline, new business, excludes test companies and 'Port'.

In [ ]:
import snowflake.connector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Connect using default connection (externalbrowser auth - a browser tab will open for SSO)
conn = snowflake.connector.connect(connection_name="default")
cur = conn.cursor()
cur.execute("USE SCHEMA PORT_ANALYTICS_PROD.DWH")

def run_query(sql):
    cur.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def quarter_label(dt):
    return pd.Timestamp(dt).to_period('Q').strftime('%YQ%q')

print(f"Connected to: {conn.account}")

## 1. Region / Team Concentration

In [ ]:
region_team_data = run_query("""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(f.DEAL_TEAM_NAME, 'Unassigned') AS team,
    COUNT(*) AS deals_lost,
    SUM(COALESCE(f.DEAL_TOTAL_ARR, 0)) AS arr_lost
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1, 2
ORDER BY 1, 2
""")
region_team_data['CLOSE_QUARTER'] = pd.to_datetime(region_team_data['CLOSE_QUARTER'])
region_team_data['qlabel'] = region_team_data['CLOSE_QUARTER'].apply(quarter_label)

top_teams = ['US MM', 'US Ent', 'EMEA MM', 'EMEA Ent', 'APJ', 'Unassigned']
df_team_top = region_team_data[region_team_data['TEAM'].isin(top_teams)]

pivot = df_team_top.pivot_table(index='qlabel', columns='TEAM', values='DEALS_LOST', fill_value=0)
pivot = pivot.reindex(sorted(pivot.index))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot[top_teams].plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10')
axes[0].set_title('Closed Lost by Team (Absolute)', fontweight='bold')
axes[0].set_ylabel('# Deals Lost')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=7, loc='upper left')

pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct[top_teams].plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10')
axes[1].set_title('Closed Lost by Team (% Mix)', fontweight='bold')
axes[1].set_ylabel('% of Deals Lost')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=7, loc='upper left')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

## 2. Dead Deals / Long Dwell Time (by Segment)

In [ ]:
dwell_time_data = run_query("""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(c.SEGMENT, 'Other') AS segment,
    COUNT(*) AS total_deals,
    AVG(DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE)) AS avg_dwell_days,
    MEDIAN(DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE)) AS median_dwell_days,
    COUNT(CASE WHEN DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE) > 180 THEN 1 END) AS dead_deals_180d,
    ROUND(100.0 * COUNT(CASE WHEN DATEDIFF('day', f.DEAL_CREATED_DATE, f.DEAL_CLOSED_DATE) > 180 THEN 1 END) / COUNT(*), 1) AS pct_dead_180d
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1, 2
ORDER BY 1, 2
""")
dwell_time_data['CLOSE_QUARTER'] = pd.to_datetime(dwell_time_data['CLOSE_QUARTER'])
dwell_time_data['qlabel'] = dwell_time_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (seg, color) in enumerate([('Enterprise', '#e74c3c'), ('Mid-Market', '#3498db')]):
    df_seg = dwell_time_data[dwell_time_data['SEGMENT'] == seg].sort_values('CLOSE_QUARTER')

    axes[0, i].plot(df_seg['qlabel'], df_seg['AVG_DWELL_DAYS'], marker='o', label='Avg Dwell', color=color)
    axes[0, i].plot(df_seg['qlabel'], df_seg['MEDIAN_DWELL_DAYS'], marker='s', label='Median Dwell', color=color, alpha=0.5, linestyle='--')
    axes[0, i].set_title(f'{seg} — Dwell Time (days)', fontweight='bold')
    axes[0, i].set_ylabel('Days')
    axes[0, i].tick_params(axis='x', rotation=45)
    axes[0, i].legend(fontsize=8)
    axes[0, i].grid(axis='y', alpha=0.3)

    axes[1, i].bar(df_seg['qlabel'], df_seg['DEAD_DEALS_180D'], color=color, alpha=0.7, label='Dead (>180d)')
    ax2 = axes[1, i].twinx()
    ax2.plot(df_seg['qlabel'], df_seg['PCT_DEAD_180D'], marker='D', color='#2c3e50', linewidth=2, label='% of total')
    axes[1, i].set_title(f'{seg} — Dead Deals (>180d)', fontweight='bold')
    axes[1, i].set_ylabel('# Dead Deals')
    ax2.set_ylabel('% of Total')
    axes[1, i].tick_params(axis='x', rotation=45)
    lines1, labels1 = axes[1, i].get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    axes[1, i].legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.tight_layout()
plt.show()

## 3. ARR Shrinkage (Amount Changes in Deal History)

In [ ]:
arr_shrinkage_data = run_query("""
WITH amount_changes AS (
    SELECT
        h.DEALID,
        FIRST_VALUE(TRY_TO_DECIMAL(h.VALUE, 15, 2)) OVER (PARTITION BY h.DEALID ORDER BY h.TIMESTAMP ASC) AS first_amount,
        FIRST_VALUE(TRY_TO_DECIMAL(h.VALUE, 15, 2)) OVER (PARTITION BY h.DEALID ORDER BY h.TIMESTAMP DESC) AS last_amount,
        COUNT(*) OVER (PARTITION BY h.DEALID) AS num_changes
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY h
    WHERE h.PROPERTY = 'amount'
      AND TRY_TO_DECIMAL(h.VALUE, 15, 2) IS NOT NULL
),
deduped AS (
    SELECT DISTINCT DEALID, first_amount, last_amount, num_changes
    FROM amount_changes
    WHERE first_amount > 0
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_deals_with_history,
    AVG(d.first_amount) AS avg_first_amount,
    AVG(d.last_amount) AS avg_last_amount,
    AVG(d.last_amount - d.first_amount) AS avg_arr_change,
    ROUND(AVG(CASE WHEN d.first_amount > 0 THEN (d.last_amount / d.first_amount) * 100 END), 1) AS avg_pct_of_original,
    SUM(CASE WHEN d.last_amount < d.first_amount THEN 1 ELSE 0 END) AS deals_shrank,
    ROUND(100.0 * SUM(CASE WHEN d.last_amount < d.first_amount THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_deals_shrank
FROM FACT_DEALS f
INNER JOIN deduped d ON d.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER)
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1
ORDER BY 1
""")
arr_shrinkage_data['CLOSE_QUARTER'] = pd.to_datetime(arr_shrinkage_data['CLOSE_QUARTER'])
arr_shrinkage_data['qlabel'] = arr_shrinkage_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(arr_shrinkage_data['qlabel'], arr_shrinkage_data['AVG_FIRST_AMOUNT'] / 1000, marker='o', color='#27ae60', label='Avg First Amount ($K)')
axes[0].plot(arr_shrinkage_data['qlabel'], arr_shrinkage_data['AVG_LAST_AMOUNT'] / 1000, marker='s', color='#e74c3c', label='Avg Last Amount ($K)')
axes[0].set_title('ARR Progression: First vs Final Amount', fontweight='bold')
axes[0].set_ylabel('Amount ($K)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

ax1 = axes[1]
ax1.bar(arr_shrinkage_data['qlabel'], arr_shrinkage_data['DEALS_SHRANK'], color='#e74c3c', alpha=0.7, label='# Deals Shrank')
ax1b = ax1.twinx()
ax1b.plot(arr_shrinkage_data['qlabel'], arr_shrinkage_data['PCT_DEALS_SHRANK'], marker='D', color='#2c3e50', linewidth=2, label='% Shrank')
ax1.set_title('Deals with ARR Shrinkage (first > final)', fontweight='bold')
ax1.set_ylabel('# Deals')
ax1b.set_ylabel('% of Deals with History')
ax1.tick_params(axis='x', rotation=45)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.tight_layout()
plt.show()

## 4. Owner Changes (Post-Demo)

In [ ]:
owner_changes_data = run_query("""
WITH stage_exit AS (
    SELECT DEALID, MIN(TIMESTAMP) AS post_sdr_ts
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY
    WHERE PROPERTY = 'dealstage'
      AND VALUE NOT IN ('65800978', '134696621', 'closedlost', 'closedwon')
    GROUP BY 1
),
owner_changes AS (
    SELECT
        h.DEALID,
        COUNT(*) AS owner_change_count
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY h
    INNER JOIN stage_exit se ON se.DEALID = h.DEALID
    WHERE h.PROPERTY = 'hubspot_owner_id'
      AND h.TIMESTAMP >= se.post_sdr_ts
    GROUP BY 1
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_deals,
    COUNT(oc.DEALID) AS deals_with_owner_change,
    ROUND(100.0 * COUNT(oc.DEALID) / COUNT(*), 1) AS pct_owner_changed,
    AVG(COALESCE(oc.owner_change_count, 0)) AS avg_owner_changes,
    SUM(CASE WHEN COALESCE(oc.owner_change_count, 0) >= 2 THEN 1 ELSE 0 END) AS deals_with_2plus_changes
FROM FACT_DEALS f
LEFT JOIN owner_changes oc ON oc.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER)
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1
ORDER BY 1
""")
owner_changes_data['CLOSE_QUARTER'] = pd.to_datetime(owner_changes_data['CLOSE_QUARTER'])
owner_changes_data['qlabel'] = owner_changes_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(owner_changes_data['qlabel'], owner_changes_data['PCT_OWNER_CHANGED'], marker='o', color='#e67e22', linewidth=2)
axes[0].fill_between(range(len(owner_changes_data)), owner_changes_data['PCT_OWNER_CHANGED'], alpha=0.2, color='#e67e22')
axes[0].set_title('% of Lost Deals with Owner Change (post-SDR)', fontweight='bold')
axes[0].set_ylabel('%')
axes[0].set_xticks(range(len(owner_changes_data)))
axes[0].set_xticklabels(owner_changes_data['qlabel'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(owner_changes_data['qlabel'], owner_changes_data['DEALS_WITH_2PLUS_CHANGES'], color='#c0392b', alpha=0.7)
axes[1].set_title('Deals with 2+ Owner Changes (post-SDR)', fontweight='bold')
axes[1].set_ylabel('# Deals')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Competitive Losses

In [ ]:
competitive_data = run_query("""
WITH evaluated_comp AS (
    SELECT DISTINCT DEALID
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY
    WHERE PROPERTY IN ('evaluated_competitors', 'competition')
      AND VALUE IS NOT NULL AND VALUE != '' AND LOWER(VALUE) NOT IN ('none', 'unknown')
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_lost,
    SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' THEN 1 ELSE 0 END) AS reason_competitive,
    COUNT(ec.DEALID) AS had_evaluated_competitor,
    SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' OR ec.DEALID IS NOT NULL THEN 1 ELSE 0 END) AS competitive_losses_combined,
    ROUND(100.0 * SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' OR ec.DEALID IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_competitive,
    SUM(CASE WHEN f.CLOSED_LOST_REASON ILIKE '%compet%' OR ec.DEALID IS NOT NULL THEN COALESCE(f.DEAL_TOTAL_ARR, 0) ELSE 0 END) AS competitive_arr_lost
FROM FACT_DEALS f
LEFT JOIN evaluated_comp ec ON ec.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER)
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1
ORDER BY 1
""")
competitive_data['CLOSE_QUARTER'] = pd.to_datetime(competitive_data['CLOSE_QUARTER'])
competitive_data['qlabel'] = competitive_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.bar(competitive_data['qlabel'], competitive_data['COMPETITIVE_LOSSES_COMBINED'], color='#e74c3c', alpha=0.7, label='Combined Competitive')
ax1.bar(competitive_data['qlabel'], competitive_data['REASON_COMPETITIVE'], color='#f39c12', alpha=0.9, label='Reason only')
ax1b = ax1.twinx()
ax1b.plot(competitive_data['qlabel'], competitive_data['PCT_COMPETITIVE'], marker='D', color='#2c3e50', linewidth=2, label='% of Total')
ax1.set_title('Competitive Losses (Reason + Evaluated Competitor)', fontweight='bold')
ax1.set_ylabel('# Competitive Losses')
ax1b.set_ylabel('% of Total Lost')
ax1.tick_params(axis='x', rotation=45)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

axes[1].bar(competitive_data['qlabel'], competitive_data['COMPETITIVE_ARR_LOST'].astype(float) / 1_000_000, color='#8e44ad', alpha=0.7)
axes[1].set_title('Competitive ARR Lost ($M)', fontweight='bold')
axes[1].set_ylabel('ARR Lost ($M)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Mega Source Mix Shift

In [ ]:
mega_source_data = run_query("""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(f.MEGA_SOURCE, 'Unknown') AS mega_source,
    COUNT(*) AS deals_lost,
    SUM(COALESCE(f.DEAL_TOTAL_ARR, 0)) AS arr_lost
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1, 2
ORDER BY 1, 2
""")
mega_source_data['CLOSE_QUARTER'] = pd.to_datetime(mega_source_data['CLOSE_QUARTER'])
mega_source_data['qlabel'] = mega_source_data['CLOSE_QUARTER'].apply(quarter_label)

sources = ['Inbound', 'Outbount SDR', 'Outbound Sales', 'Channel', 'Referrals', 'Unknown']
pivot_src = mega_source_data.pivot_table(index='qlabel', columns='MEGA_SOURCE', values='DEALS_LOST', fill_value=0)
pivot_src = pivot_src.reindex(sorted(pivot_src.index))
cols = [c for c in sources if c in pivot_src.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_src[cols].plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2')
axes[0].set_title('Closed Lost by Mega Source (Absolute)', fontweight='bold')
axes[0].set_ylabel('# Deals Lost')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=7)

pivot_src_pct = pivot_src.div(pivot_src.sum(axis=1), axis=0) * 100
pivot_src_pct[cols].plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('Closed Lost by Mega Source (% Mix)', fontweight='bold')
axes[1].set_ylabel('% of Deals Lost')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=7)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

## 7. Qualification Bar Loosening

In [ ]:
qualification_data = run_query("""
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COUNT(*) AS total_deals,
    COUNT(CASE WHEN f.QUALIFIED_DATE IS NOT NULL THEN 1 END) AS qualified_deals,
    ROUND(100.0 * COUNT(CASE WHEN f.QUALIFIED_DATE IS NOT NULL THEN 1 END) / COUNT(*), 1) AS pct_qualified,
    AVG(CASE WHEN f.QUALIFIED_DATE IS NOT NULL THEN DATEDIFF('day', f.DEAL_CREATED_DATE, f.QUALIFIED_DATE) END) AS avg_days_to_qualify,
    COUNT(CASE WHEN f.QUALIFIED_DATE IS NOT NULL AND DATEDIFF('day', f.DEAL_CREATED_DATE, f.QUALIFIED_DATE) <= 3 THEN 1 END) AS fast_qualified_3d,
    ROUND(100.0 * COUNT(CASE WHEN f.QUALIFIED_DATE IS NOT NULL AND DATEDIFF('day', f.DEAL_CREATED_DATE, f.QUALIFIED_DATE) <= 3 THEN 1 END)
          / NULLIF(COUNT(CASE WHEN f.QUALIFIED_DATE IS NOT NULL THEN 1 END), 0), 1) AS pct_fast_qualified
FROM FACT_DEALS f
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1
ORDER BY 1
""")
qualification_data['CLOSE_QUARTER'] = pd.to_datetime(qualification_data['CLOSE_QUARTER'])
qualification_data['qlabel'] = qualification_data['CLOSE_QUARTER'].apply(quarter_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.bar(qualification_data['qlabel'], qualification_data['PCT_QUALIFIED'], color='#27ae60', alpha=0.7, label='% Qualified')
ax1b = ax1.twinx()
ax1b.plot(qualification_data['qlabel'], qualification_data['AVG_DAYS_TO_QUALIFY'], marker='o', color='#e74c3c', linewidth=2, label='Avg Days to Qualify')
ax1.set_title('Qualification Rate & Speed (Lost Deals)', fontweight='bold')
ax1.set_ylabel('% of Lost Deals Qualified')
ax1b.set_ylabel('Avg Days to Qualify')
ax1.tick_params(axis='x', rotation=45)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

axes[1].plot(qualification_data['qlabel'], qualification_data['PCT_FAST_QUALIFIED'], marker='s', color='#8e44ad', linewidth=2)
axes[1].fill_between(range(len(qualification_data)), qualification_data['PCT_FAST_QUALIFIED'], alpha=0.15, color='#8e44ad')
axes[1].set_title('% of Qualified Deals Qualified in <=3 Days', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].set_xticks(range(len(qualification_data)))
axes[1].set_xticklabels(qualification_data['qlabel'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Stage Before Close Mix Shift

In [ ]:
stage_before_close_data = run_query("""
WITH last_stage AS (
    SELECT
        h.DEALID,
        h.VALUE AS stage_before_close,
        ROW_NUMBER() OVER (PARTITION BY h.DEALID ORDER BY h.TIMESTAMP DESC) AS rn
    FROM RAW_PORT_EXTERNAL.AB_HUBSPOT_HISTORY.DEALS_PROPERTY_HISTORY h
    WHERE h.PROPERTY = 'dealstage'
      AND h.VALUE != 'closedlost'
      AND h.VALUE != 'closedwon'
)
SELECT
    DATE_TRUNC('quarter', f.DEAL_CLOSED_DATE)::DATE AS close_quarter,
    COALESCE(ls.stage_before_close, 'Unknown') AS stage_before_close,
    COUNT(*) AS deals_lost
FROM FACT_DEALS f
LEFT JOIN last_stage ls ON ls.DEALID = TRY_CAST(f.DEAL_CRM_ID AS NUMBER) AND ls.rn = 1
LEFT JOIN DIM_COMPANY c ON c.SK_COMPANY = f.SK_COMPANY
WHERE f.IS_CLOSED = TRUE
  AND f.IS_WON = FALSE
  AND f.ARCHIVED = FALSE
  AND f._DELETED_TIMESTAMP IS NULL
  AND f.PIPELINE = 'Classic'
  AND f.DEAL_TYPE = 'newbusiness'
  AND f.DEAL_CLOSED_DATE >= '2023-01-01'
  AND COALESCE(c.COMPANY_NAME, '') NOT ILIKE '%test%'
  AND COALESCE(c.COMPANY_NAME, '') != 'Port'
GROUP BY 1, 2
ORDER BY 1, 3 DESC
""")
stage_before_close_data['CLOSE_QUARTER'] = pd.to_datetime(stage_before_close_data['CLOSE_QUARTER'])
stage_before_close_data['qlabel'] = stage_before_close_data['CLOSE_QUARTER'].apply(quarter_label)

stage_map = {
    '65800978': 'SDR Discovery',
    '65800980': 'Demo / Presentation',
    '982622489': 'Business Validation',
    '65537604': 'Formal Pilot',
    '134696621': 'SDR / Omitted Opp',
    '22339760': 'Opportunity',
    '65537605': 'Business Case Confirmation',
    '134696626': 'Negotiation / Legal',
    '134696624': 'Formal Pilot (alt)',
    '134696623': 'Business Validation (alt)',
    '134696622': 'Demo (alt)',
    'contractsent': 'Contract Sent',
    'Unknown': 'Unknown'
}
stage_before_close_data['stage_label'] = stage_before_close_data['STAGE_BEFORE_CLOSE'].map(stage_map).fillna(stage_before_close_data['STAGE_BEFORE_CLOSE'])

pivot_stage = stage_before_close_data.pivot_table(index='qlabel', columns='stage_label', values='DEALS_LOST', fill_value=0)
pivot_stage = pivot_stage.reindex(sorted(pivot_stage.index))

main_stages = [c for c in ['SDR Discovery', 'SDR / Omitted Opp', 'Demo / Presentation',
               'Business Validation', 'Formal Pilot', 'Business Case Confirmation',
               'Negotiation / Legal', 'Unknown'] if c in pivot_stage.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_stage[main_stages].plot(kind='bar', stacked=True, ax=axes[0], colormap='viridis')
axes[0].set_title('Stage Before Close-Lost (Absolute)', fontweight='bold')
axes[0].set_ylabel('# Deals Lost')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=7, loc='upper left')

pivot_stage_pct = pivot_stage.div(pivot_stage.sum(axis=1), axis=0) * 100
pivot_stage_pct[main_stages].plot(kind='bar', stacked=True, ax=axes[1], colormap='viridis')
axes[1].set_title('Stage Before Close-Lost (% Mix)', fontweight='bold')
axes[1].set_ylabel('% of Deals Lost')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=7, loc='upper left')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

print('\nNote: "Unknown" = deals without stage-change history in DEALS_PROPERTY_HISTORY.'
      '\nHistory coverage improves for recent quarters (2025+).')